Split data into train, test and validation, using a csv file containing
* Path:
* Video:

In [1]:
%load_ext autoreload
%autoreload 2

# %autoreload 1: Reloads only modules imported with %aimport.
# %autoreload 2: Reloads all modules every time you execute a cell. This is the most popular setting for active development.

# 1. Split the data into train, test, validation
Split on file path first.

In [2]:
import torch
from torch.utils.data import DataLoader, random_split
import numpy as np
import os
import json
from pathlib import Path
from sklearn.model_selection import train_test_split
import pandas as pd

SEED = 42 # Set global seed for any random generator

## 1.1 Split at video level

In [14]:
# Step 1 - get unique video identifiers from frame file names
# Assumeing filenames follow a pattern like "IMG_0158_frame_000218.png"
fn_list = '/home/hice1/wzhu97/scratch/2026_freeman_bird_behavior/data/wz_data/H23_frame_list.subset_10_videos.csv'
df_file_list = pd.read_csv(fn_list, header=None).rename(columns={0:'Path', 1:'Video'})
print('# Get unique video IDs from frame file names')
display(df_file_list.head(2))
video_names = df_file_list['Video'].unique()
print(f"Total unique videos: {df_file_list['Video'].nunique()}")

# Step 2 - split at video level
train_frac, test_val_frac = 0.7, 0.3 # Fractions to use for training and testing (plus validation if necessary, split 50/50 for test and validation)
train_videos, temp_videos = train_test_split(video_names, train_size=train_frac, random_state=SEED)
val_videos, test_videos = train_test_split(temp_videos, test_size=0.5, random_state=SEED)

print(f"# Train videos: {len(train_videos)}")
print(f"# Val videos: {len(val_videos)}")
print(f"# Test videos: {len(test_videos)}")

# Step 3 - map back to frame paths
train_frames = df_file_list[df_file_list['Video'].isin(train_videos)]['Path']
val_frames = df_file_list[df_file_list['Video'].isin(val_videos)]['Path']
test_frames = df_file_list[df_file_list['Video'].isin(test_videos)]['Path']

print(f"\n# Train frames: {len(train_frames)}")
print(f"# Val frames: {len(val_frames)}")
print(f"# Test frames: {len(test_frames)}")

# Get unique video IDs from frame file names


,Path,Video
0,/home/hice1/wzhu97/scratch/2026_freeman_bird_b...,IMG_0067
1,/home/hice1/wzhu97/scratch/2026_freeman_bird_b...,IMG_0067


Total unique videos: 9
# Train videos: 6
# Val videos: 1
# Test videos: 2

# Train frames: 1818
# Val frames: 303
# Test frames: 606


In [15]:
test_frames

303     /home/hice1/wzhu97/scratch/2026_freeman_bird_b...
304     /home/hice1/wzhu97/scratch/2026_freeman_bird_b...
305     /home/hice1/wzhu97/scratch/2026_freeman_bird_b...
306     /home/hice1/wzhu97/scratch/2026_freeman_bird_b...
307     /home/hice1/wzhu97/scratch/2026_freeman_bird_b...
                              ...                        
2419    /home/hice1/wzhu97/scratch/2026_freeman_bird_b...
2420    /home/hice1/wzhu97/scratch/2026_freeman_bird_b...
2421    /home/hice1/wzhu97/scratch/2026_freeman_bird_b...
2422    /home/hice1/wzhu97/scratch/2026_freeman_bird_b...
2423    /home/hice1/wzhu97/scratch/2026_freeman_bird_b...
Name: Path, Length: 606, dtype: str

## 1.2 Build a PyTorch Dataset from the frame paths

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

class BirdFrameDataset(Dataset):
    def __init__(self, frame_paths, annotations, transforms=None):
        '''
        frame_paths: array of paths
        '''
        self.frame_paths = frame_paths
        self.annotations = annotations # your COCO annotations dict (loaded from the json file, such as coco['annotations'])
        self.transforms = transforms
        
        # build a lookup from filename to annotations
        # If the image is not found in annotation then skip this one
        self.fname_to_annos = {
            img['file_name']: [
                ann for ann in annotations['annotations'] 
                if ann['image_id'] == img['id']
            ]
            for img in annotations['images']
        }

    def __len__(self):
        '''
        Return number of frames of this dataset
        '''
        return len(self.frame_paths)

    def __getitem__(self, idx):
        '''
        Get loaded image and annotation of the i-th item
        '''
        path = self.frame_paths[idx]
        image = Image.open(path).convert('RGB')
        annos = self.fname_to_annos.get(path.name, [])   # empty list if no birds
        
        if self.transforms:
            image = self.transforms(image)
            
        return image, annos

# Define transforms
# Composes several transforms together.
# In this code I used:
# - Resize to the size of frames detected in the annotations
# - Convert the image to tensor
# - Normalize a tensor image with mean and standard deviation
transforms = T.Compose([
    T.Resize((1280, 720)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], 
                std=[0.229, 0.224, 0.225])   # ImageNet normalization, standard for DETR
])

coco_fn = '/home/hice1/wzhu97/scratch/2026_freeman_bird_behavior/data/wz_data/H23/Annotations/instances_merged.json'
with open(coco_fn) as f:
    coco = json.load(f)

# create datasets
train_dataset = BirdFrameDataset(train_frames, coco, transforms)
val_dataset   = BirdFrameDataset(val_frames,   coco, transforms)
test_dataset  = BirdFrameDataset(test_frames,  coco, transforms)

# create dataloaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset,  batch_size=4, shuffle=False, num_workers=4)

In [3]:
path = Path('/home/hice1/wzhu97/scratch/2026_freeman_bird_behavior/data/wz_data/H23/frames/IMG_0158_frame_000218.png')

In [11]:
path.parent

PosixPath('/home/hice1/wzhu97/scratch/2026_freeman_bird_behavior/data/wz_data/H23/frames')

# 2. Package the code above and test

In [26]:
# Define transforms
# Composes several transforms together.
# In this code I used:
# - Resize to the size of frames detected in the annotations
# - Convert the image to tensor
# - Normalize a tensor image with mean and standard deviation
# transforms = T.Compose([
#     T.Resize((1280, 720)),
#     T.ToTensor(),
#     T.Normalize(mean=[0.485, 0.456, 0.406], 
#                 std=[0.229, 0.224, 0.225])   # ImageNet normalization, standard for DETR
# ])
# coco_fn = '/home/hice1/wzhu97/scratch/2026_freeman_bird_behavior/data/wz_data/H23/Annotations/instances_merged.json'
# with open(coco_fn) as f:
#     coco = json.load(f)
    
# transforms = T.Compose([
#     T.ToTensor(),
#     T.Normalize(mean=[0.485, 0.456, 0.406], 
#                 std=[0.229, 0.224, 0.225])   # ImageNet normalization, standard for DETR
# ])

# # create datasets
# train_dataset = BirdFrameDataset(train_frames, coco, transforms)
# val_dataset   = BirdFrameDataset(val_frames,   coco, transforms)
# test_dataset  = BirdFrameDataset(test_frames,  coco, transforms)

# # create dataloaders
# train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,  num_workers=4)
# val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False, num_workers=4)
# test_loader  = DataLoader(test_dataset,  batch_size=4, shuffle=False, num_workers=4)

In [1]:
from utilities.dataset import create_dataloaders

coco_fn = '/home/hice1/wzhu97/scratch/2026_freeman_bird_behavior/data/wz_data/H23/Annotations/instances_merged.json'
fn_frame_list = '/home/hice1/wzhu97/scratch/2026_freeman_bird_behavior/data/wz_data/H23_frame_list.subset_0.01.csv' 

# Split the data into train, test, validation
# Perform transformations: resizing, normalization
# num_workers: Enables multi-process data loading 
train_loader, val_loader, test_loader = create_dataloaders(frames_csv=fn_frame_list,
                                                           annotation_path=coco_fn, 
                                                           test_size=0.2, val_size=0.1,
                                                           batch_size=2, num_workers=4,
                                                           random_state=42)



# Frames with >1 annotations were excluded: N=32

# ##### Data split summary #####
# Split      Videos   Frames
# Train          60      195
# Val            18       57
# Test            9       27
# Total          87      279


In [5]:
# Verify the split worked as expected (should have 279 frames in the subset file)
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

# check one batch loads correctly
images, annotations = next(iter(train_loader))
print(f"Batch image shape: {images.shape}")   # should be [2, 3, 720, 1280]
print(f"Annotations: {annotations}")

Train batches: 98
Val batches:   29
Test batches:  14
Batch image shape: torch.Size([2, 3, 720, 1280])
Annotations: [[], [{'id': 1490, 'image_id': 6556, 'category_id': 1, 'segmentation': [], 'area': 3692.503999999995, 'bbox': [1035.2, 522.39, 68.89, 53.6], 'iscrowd': 0, 'attributes': {'occluded': False, 'rotation': 0.0, 'track_id': 0, 'keyframe': False}}]]
